# PneumoFusionNet — Phase 2: Multimodal Classification
## Image + Radiology Report (Impression) Fusion

### Project Overview
Phase 2 extends the image-only baseline (Phase 1) by incorporating **radiology report text** (Impression section).  
Both modalities are fused via concatenation and passed through a shared classifier.

The model combines:

- **Image Branch** → PneumoFusionNet (ResNet50 + DSC + GCSA) → 1024-d  
- **Text Branch**  → BioClinicalBERT [CLS] token → 768-d  
- **Fusion**       → Concatenate [1024 + 768] = 1792-d → Classifier

---

## Dataset

- MIMIC-CXR JPG (P10 folder — 139 labelled samples)
- Labels: NORMAL (No Finding = 1.0) / PNEUMONIA (Pneumonia = 1.0)
- Text: IMPRESSION section of each radiology report
- Split: **80% Train / 20% Test (stratified)**

---

## Notebook Structure

1. Environment Setup  
2. Dataset Loading  
3. 80/20 Train/Test Split  
4. Transforms & Tokeniser  
5. Dataset Class & DataLoaders  
6. Model Architecture (Image + Text + Fusion)  
7. Loss, Optimiser & Scheduler  
8. Training  
9. Evaluation & Metrics  
10. Save Features for Phase 3  

---

## Author
IIT Guwahati  
B.Sc. in Data Science & Artificial Intelligence  
📧 Email: ay346185@gmail.com

## 1. Environment Setup

In [1]:
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR        = os.path.abspath(os.path.join(os.getcwd(), '..'))
CSV_PATH        = os.path.join(BASE_DIR, 'dataset_139', 'mimic_multimodal_dataset_v2.csv')
PHASE1_WEIGHTS  = os.path.join(BASE_DIR, 'outputs', 'best_pneumofusion_mimic.pth')
SAVE_DIR        = os.path.join(BASE_DIR, 'outputs')
MODEL_PATH      = os.path.join(SAVE_DIR, 'best_phase2_multimodal.pth')
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE=224; BATCH_SIZE=8; NUM_EPOCHS=30; LR=1e-4; WEIGHT_DECAY=1e-4; NUM_CLASSES=2
MAX_TEXT_LEN=128; BERT_MODEL='emilyalsentzer/Bio_ClinicalBERT'
CLASSES=['NORMAL','PNEUMONIA']; MEAN=[0.485]; STD=[0.229]

print('BASE_DIR :', BASE_DIR)
print('CSV      :', CSV_PATH)
print('Exists   :', os.path.exists(CSV_PATH))
print('BERT     :', BERT_MODEL)import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score, auc
)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR        = os.path.abspath(os.path.join(os.getcwd(), '..'))
CSV_PATH        = os.path.join(BASE_DIR, 'dataset_139', 'mimic_multimodal_dataset_v2.csv')
PHASE1_WEIGHTS  = os.path.join(BASE_DIR, 'outputs', 'best_pneumofusion_mimic.pth')
SAVE_DIR        = os.path.join(BASE_DIR, 'outputs')
MODEL_PATH      = os.path.join(SAVE_DIR, 'best_phase2_multimodal.pth')
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE=224; BATCH_SIZE=8; NUM_EPOCHS=30; LR=1e-4; WEIGHT_DECAY=1e-4; NUM_CLASSES=2
MAX_TEXT_LEN=128; BERT_MODEL='emilyalsentzer/Bio_ClinicalBERT'
CLASSES=['NORMAL','PNEUMONIA']; MEAN=[0.485]; STD=[0.229]

print('BASE_DIR :', BASE_DIR)
print('CSV      :', CSV_PATH)
print('Exists   :', os.path.exists(CSV_PATH))
print('BERT     :', BERT_MODEL)

BASE_DIR : C:\2026\PneumoFusionNet\mimic\mimic_pilot
CSV      : C:\2026\PneumoFusionNet\mimic\mimic_pilot\dataset_139\mimic_multimodal_dataset_v2.csv
Exists   : True
BERT     : emilyalsentzer/Bio_ClinicalBERT


In [7]:
df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print(df['label_name'].value_counts())
df.head(10)

Shape: (139, 7)
Normal       118
Pneumonia     21
Name: label_name, dtype: int64


,subject_id,study_id,image_path,label,label_name,report_path,impression
0,10000032,50414267,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,No acute cardiopulmonary process.
1,10000032,53189527,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,No acute cardiopulmonary abnormality.
2,10000032,53911762,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,No acute intrathoracic process.
3,10000032,56699142,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,No acute cardiopulmonary process.
4,10000898,50771383,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,No acute intrathoracic process.
5,10000898,54205396,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,No evidence of acute cardiopulmonary process.
6,10000935,50578979,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,1,Pneumonia,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,1. Low lung volumes and mild pulmonary vascula...
7,10000935,55697293,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,Stable chest radiograph.
8,10000980,51967283,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,1,Pneumonia,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,"Right upper lobe pneumonia or mass. However, g..."
9,10000980,54577367,C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1...,0,Normal,C:\2026\PneumoFusionNet\mimic\mimic_pilot\repo...,No radiographic evidence for pneumonia.


In [8]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print('Train:', len(train_df), '| Pneumonia:', train_df.label.sum(), '| Normal:', (train_df.label==0).sum())
print('Test :', len(test_df),  '| Pneumonia:', test_df.label.sum(),  '| Normal:', (test_df.label==0).sum())

Train: 111 | Pneumonia: 17 | Normal: 94
Test : 28 | Pneumonia: 4 | Normal: 24


In [9]:
tfms = {
    'train': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((256, 256)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.25, contrast=0.25),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
    'test': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
}
print('Image transforms ready.')

Image transforms ready.


In [10]:
print('Loading BioClinicalBERT tokeniser...')
tokeniser = AutoTokenizer.from_pretrained(BERT_MODEL)
print('Tokeniser loaded.')

# Quick test
sample_enc = tokeniser(
    'No acute cardiopulmonary process.',
    return_tensors='pt',
    max_length=MAX_TEXT_LEN,
    padding='max_length',
    truncation=True
)
print('Token IDs shape:', sample_enc['input_ids'].shape)

Loading BioClinicalBERT tokeniser...
Tokeniser loaded.
Token IDs shape: torch.Size([1, 128])


In [11]:
class MIMICMultimodalDataset(Dataset):
    """Returns (image_tensor, input_ids, attention_mask, label)"""
    def __init__(self, df, tokeniser, transform=None, max_len=MAX_TEXT_LEN):
        self.df        = df.reset_index(drop=True)
        self.tokeniser = tokeniser
        self.transform = transform
        self.max_len   = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Image ──────────────────────────────────────────────
        img = Image.open(row.image_path)
        if self.transform: img = self.transform(img)

        # ── Text (impression) ──────────────────────────────────
        text = str(row.impression) if pd.notna(row.impression) else 'no impression available'
        enc  = self.tokeniser(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)

        return img, input_ids, attention_mask, int(row.label)

print('MIMICMultimodalDataset class defined.')

MIMICMultimodalDataset class defined.


In [13]:
# WeightedRandomSampler — handles class imbalance
